In [7]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

base = '../../data/'
train = pd.read_csv(base + 'train.csv')
test  = pd.read_csv(base + 'test.csv')

# 1. Feature Engineering (Missing flags & Age groups)
missing_cols = ['followup_days','hemoglobin_g_dl','creatinine_mg_dl',
                'sodium_mmol_l','heart_rate_bpm','systolic_bp_mmhg']

for df in [train, test]:
    df['n_missing'] = df[missing_cols].isna().sum(axis=1)
    for c in missing_cols:
        df[c + '_flag'] = df[c].isna().astype(int)
    df['age_group'] = pd.cut(df['age'], bins=[0, 30, 50, 70, 120],
                             labels=['<30', '30-50', '50-70', '70+'])

num_cols = ['age', 'socioeconomic_index', 'prior_admissions_12m', 'comorbidity_count',
            'length_of_stay_days', 'medication_count', 'missed_appointments_12m',
            'followup_days', 'hemoglobin_g_dl', 'creatinine_mg_dl', 'sodium_mmol_l',
            'heart_rate_bpm', 'systolic_bp_mmhg', 'n_missing'] + \
           [c + '_flag' for c in missing_cols]

cat_cols = ['sex', 'rurality', 'hospital_type', 'region', 'discharge_disposition', 'care_pathway']

# 2. Preprocessing Pipeline
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')) 
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

# 3. Logistic Regression Model
# WHY: Standard interpretable baseline. We use default C=1.0 to keep probabilities naturally calibrated.
clf = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', n_jobs=-1, random_state=42)
model = Pipeline([('pre', preprocessor), ('clf', clf)])

# 4. Cross-Validation to get Out-Of-Fold (OOF) predictions
y = train['readmitted_30d']
X = train.drop(columns=['patient_id', 'readmitted_30d'])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    oof_preds[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]

EPS = 1e-15
oof_preds = np.clip(oof_preds, EPS, 1 - EPS)
PREDS = oof_preds  # <-- This feeds into the harness!

# 5. Overall OOF Metrics
print(f"Logistic Regression OOF Log Loss : {log_loss(y, PREDS):.6f}")
print(f"Logistic Regression OOF Brier    : {brier_score_loss(y, PREDS):.6f}")
print(f"Logistic Regression OOF ROC-AUC  : {roc_auc_score(y, PREDS):.6f}")
print(f"Constant Baseline Log Loss was   : 0.378435")

Logistic Regression OOF Log Loss : 0.352613
Logistic Regression OOF Brier    : 0.103179
Logistic Regression OOF ROC-AUC  : 0.685290
Constant Baseline Log Loss was   : 0.378435


In [8]:
def ll(yt, pp):  return log_loss(yt, np.clip(pp, EPS, 1-EPS), labels=[0,1])
def logit(p):    p = np.clip(p, 1e-12, 1-1e-12); return np.log(p/(1-p))

def wilson_ci(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k/n; d = 1 + z**2/n
    c = (p + z**2/(2*n)) / d
    h = z*np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / d
    return (c-h, c+h)

def slice_table(col):
    rows = []
    for lev, mask in ((v, train[col] == v) for v in train[col].dropna().unique()):
        yt, pp = y[mask], PREDS[mask.values]
        lo, hi = wilson_ci(yt.sum(), len(yt))
        rows.append({
            'level': lev, 'n': int(mask.sum()), 'events': int(yt.sum()),
            'obs_rate': yt.mean(), 'rate_CI95': f"[{lo:.3f},{hi:.3f}]",
            'LogLoss': ll(yt, pp), 'Brier': brier_score_loss(yt, pp),
            'calib_gap': yt.mean() - pp.mean(),          
            'calib_intercept': logit(yt.mean()) - logit(pp.mean())
        })
    return pd.DataFrame(rows).sort_values('obs_rate', ascending=False)

def shift_report(col):
    tr = train[col].value_counts(normalize=True)
    te = test[col].value_counts(normalize=True)
    w  = (te / tr).dropna()
    weights = train[col].map(w).fillna(0.0).to_numpy()
    obs_w = np.average(y, weights=weights)
    return pd.Series({
        'train_rate': y.mean(), 'shifted_rate': obs_w,
        'gap_vs_constant': obs_w - y.mean(),
        'const_LogLoss_under_shift': -(obs_w*np.log(y.mean()) + (1-obs_w)*np.log(1-y.mean()))
    })

def abstain_report(frac=0.10, seed=42):
    pp = np.clip(PREDS, EPS, 1-EPS)
    ent = -pp*np.log(pp) - (1-pp)*np.log(1-pp)
    if np.std(ent) < 1e-12:
        rng = np.random.default_rng(seed)
        keep = rng.random(len(ent)) > frac
        note = "entropy uniform -> random 10% referred (demo only)"
    else:
        keep = ent <= np.quantile(ent, 1-frac)
        note = "top-10% entropy referred"
    return note, ll(y[keep], pp[keep]), ll(y, pp)

In [9]:
print("Q1 - Missingness Strata Performance:")
train['missing_stratum'] = pd.cut(train['n_missing'], bins=[-1,0,1,10], labels=['0 missing','1 missing','2+ missing'])
print(slice_table('missing_stratum').to_string(index=False))

print("\n" + "="*50)
print("Q4 - CARE PATHWAY SENSITIVITY TEST:")
cat_cols_no_cp = [c for c in cat_cols if c != 'care_pathway']
preprocessor_no_cp = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols_no_cp)])
model_no_cp = Pipeline([('pre', preprocessor_no_cp), ('clf', clf)])

oof_no_cp = np.zeros(len(X))
for tr_idx, val_idx in skf.split(X, y):
    model_no_cp.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    oof_no_cp[val_idx] = model_no_cp.predict_proba(X.iloc[val_idx])[:, 1]

ll_with = log_loss(y, np.clip(oof_preds, EPS, 1-EPS))
ll_without = log_loss(y, np.clip(oof_no_cp, EPS, 1-EPS))
print(f"LR WITH care_pathway    LogLoss: {ll_with:.6f}")
print(f"LR WITHOUT care_pathway LogLoss: {ll_without:.6f}")
if ll_without - ll_with < 0.002:
    print("-> care_pathway adds very little predictive value. Safe to drop for robustness!")
else:
    print("-> care_pathway adds value, but remember it shifts in the test set.")

Q1 - Missingness Strata Performance:
     level    n  events  obs_rate     rate_CI95  LogLoss    Brier  calib_gap  calib_intercept
 1 missing 1386     181  0.130592 [0.114,0.149] 0.369974 0.108604   0.001234         0.010912
 0 missing 5428     677  0.124724 [0.116,0.134] 0.348037 0.101778  -0.000146        -0.001337
2+ missing  186      23  0.123656 [0.084,0.179] 0.356798 0.103654  -0.001594        -0.014631

Q4 - CARE PATHWAY SENSITIVITY TEST:
LR WITH care_pathway    LogLoss: 0.352613
LR WITHOUT care_pathway LogLoss: 0.353296
-> care_pathway adds very little predictive value. Safe to drop for robustness!


In [10]:
print("Q2 - REGIONS:")
print(slice_table('region')[['level', 'n', 'obs_rate', 'LogLoss', 'calib_gap']].to_string(index=False))

print("\nQ3 - HOSPITAL TYPES:")
print(slice_table('hospital_type')[['level', 'n', 'obs_rate', 'LogLoss', 'calib_gap']].to_string(index=False))

print("\nQ5 - CALIBRATION GAPS (Where is the model over/under predicting?):")
for col in ['age_group', 'rurality', 'care_pathway']:
    print(f"\n[{col}]")
    print(slice_table(col)[['level', 'n', 'obs_rate', 'calib_gap', 'calib_intercept']].to_string(index=False))

print("\nQ6 - SENSITIVE SUBGROUPS:")
train['ses_quintile'] = pd.qcut(train['socioeconomic_index'], q=5, labels=['Q1','Q2','Q3','Q4','Q5'], duplicates='drop')
for col in ['sex', 'rurality', 'ses_quintile']:
    print(f"\n[{col}]")
    print(slice_table(col)[['level', 'n', 'obs_rate', 'LogLoss', 'calib_gap']].to_string(index=False))

Q2 - REGIONS:
     level    n  obs_rate  LogLoss  calib_gap
     South 1554  0.134492 0.366485   0.000305
   Central 1529  0.128842 0.349491   0.000156
      West 2591  0.124662 0.358867   0.000014
North-East 1326  0.114630 0.327736  -0.000095

Q3 - HOSPITAL TYPES:
   level    n  obs_rate  LogLoss  calib_gap
Teaching 3092  0.130013 0.365102   0.000221
District 1285  0.123735 0.339745  -0.000668
 General 2623  0.121998 0.344195   0.000304

Q5 - CALIBRATION GAPS (Where is the model over/under predicting?):

[age_group]
level    n  obs_rate  calib_gap  calib_intercept
  70+ 1477  0.222749  -0.000831        -0.004793
50-70 3150  0.120635  -0.003178        -0.029624
30-50 2030  0.078325   0.006927         0.100091
  <30  343  0.037901  -0.006425        -0.163295

[rurality]
     level    n  obs_rate  calib_gap  calib_intercept
     Rural 1243  0.156074   0.000277         0.002101
     Urban 3887  0.124260   0.000133         0.001220
Semi-urban 1870  0.109091  -0.000128        -0.001314

[ca

In [11]:
note, ll_keep, ll_all = abstain_report(0.10)
print(f"Q7 - ABSTENTION ANALYSIS ({note})")
print(f"LogLoss on kept 90% (certain patients) = {ll_keep:.6f}")
print(f"LogLoss on ALL 100% patients           = {ll_all:.6f}")
print(f"Improvement by referring 10% to humans = {ll_all - ll_keep:.6f}")

if ll_keep < ll_all:
    print("-> SUCCESS: The model knows when it is uncertain! Referring these cases improves overall reliability.")
else:
    print("-> Referral did not improve log loss. The model's confidence scores may need calibration.")

Q7 - ABSTENTION ANALYSIS (top-10% entropy referred)
LogLoss on kept 90% (certain patients) = 0.324062
LogLoss on ALL 100% patients           = 0.352613
Improvement by referring 10% to humans = 0.028551
-> SUCCESS: The model knows when it is uncertain! Referring these cases improves overall reliability.


Exception ignored in: <function ResourceTracker.__del__ at 0x7fa18b78a660>
Traceback (most recent call last):
  File "/home/teshan/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/teshan/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/teshan/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7fce28192660>
Traceback (most recent call last):
  File "/home/teshan/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/teshan/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/teshan/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__